In [ ]:
import json
import os
from collections import Counter
from urllib.parse import urlsplit

import pandas as pd

ROOT = os.getcwd()
DATASET = os.path.join(ROOT, 'final_dataset_v2.csv')
OUT = os.path.join(ROOT, 'dataset_audit_report.json')

def normalize_label(v):
    if pd.isna(v):
        return None
    s = str(v).strip().lower()
    mapping = {
        '0': 0,
        '1': 1,
        'benign': 0,
        'safe': 0,
        'legitimate': 0,
        'phishing': 1,
        'malicious': 1,
        'fraud': 1,
        'suspicious': 1,
    }
    return mapping.get(s)

df = pd.read_csv(DATASET)
df.columns = [str(c).strip().lower() for c in df.columns]
url_col = next((c for c in df.columns if 'url' in c), None)
label_col = next((c for c in df.columns if c in {'label', 'class', 'type', 'status', 'target'}), None)
if url_col is None or label_col is None:
    raise ValueError('Dataset missing URL/label columns')

subset = df[[url_col, label_col]].rename(columns={url_col: 'url', label_col: 'label'})
subset['url'] = subset['url'].astype(str).str.strip()
subset = subset[subset['url'].str.len() > 0].copy()
subset['normalized_label'] = subset['label'].map(normalize_label)
subset = subset[subset['normalized_label'].notna()].copy()
subset['normalized_label'] = subset['normalized_label'].astype(int)

def reg_domain(url):
    try:
        host = (urlsplit(url).hostname or '').lower().rstrip('.')
    except Exception:
        return ''
    if not host:
        return ''
    parts = host.split('.')
    if len(parts) >= 2:
        return '.'.join(parts[-2:])
    return host

subset['registered_domain'] = subset['url'].map(reg_domain)
exact_duplicates = int(subset.duplicated(subset=['url']).sum())
lowercase_exact_duplicates = int(subset.assign(url_lower=subset['url'].str.strip().str.lower()).duplicated(subset=['url_lower']).sum())
conflict_urls = (
    subset.assign(norm=subset['url'].str.strip().str.lower())
    .groupby('norm')['normalized_label']
    .nunique()
    .loc[lambda s: s > 1]
    .index
)
conflicting_rows = int(subset[subset['url'].str.strip().str.lower().isin(conflict_urls)].shape[0])
report = {
    'file': DATASET,
    'rows_total': int(len(subset)),
    'rows_after_valid_label': int(len(subset)),
    'label_distribution': {str(k): int(v) for k, v in sorted(Counter(subset['normalized_label'].tolist()).items())},
    'duplicate_exact_urls': exact_duplicates,
    'duplicate_normalized_urls': lowercase_exact_duplicates,
    'conflicting_label_rows': conflicting_rows,
    'unique_registered_domains': int(subset['registered_domain'].dropna().nunique()),
    'source_provenance': 'No explicit source provenance field was present in the current CSV.'
}
with open(OUT, 'w', encoding='utf-8') as f:
    json.dump(report, f, indent=2)
print(json.dumps(report, indent=2))